In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
from fundus_data_toolkit.functional import open_image
from jppype import Mosaic, vscode_theme

from fundus_odmac_toolkit.models.segmentation import segment as segment_odmac
from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models import segment_av
from fundus_vessels_toolkit.pipelines.avseg_to_tree import GNNAVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.utils.jppype import draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

## Load Image and Segment AV, OD, Macula


In [3]:
PATH = Path("/home/gaby/These/Data/Fundus/Vessels/Fundus-AVSeg/")
IMG = "036_A.png"

fundus_gt = (
    FundusData(image=PATH / "1-images" / IMG, av=PATH / "2-av" / IMG).crop_to_roi(ensure_square=True).rescale(1500)
)

odmac = segment_odmac(fundus_gt.image.transpose(1, 2, 0) * 255).argmax(axis=0).numpy(force=True)
_ = fundus_gt.update(od=odmac == 1, macula=odmac == 2, inplace=True)


In [4]:
fundus = fundus_gt.mutable_copy()
segment_av(fundus)

naive_av2tree = NaiveAVSegToTree()
av2tree = GNNAVSegToTree()

trees = av2tree(fundus)
trees_gt = naive_av2tree(fundus_gt)


m = Mosaic(2, cols_titles=["Predicted", "Ground Truth"], cell_height=400)
fundus.draw(view=m[0])
draw_trees(trees, view=m[0])
fundus_gt.draw(view=m[1])
draw_trees(trees_gt, view=m[1])
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

## Compute AV topological maps


In [5]:
from fundus_vessels_toolkit.segment_to_graph.av_map_fixing import TopologicalLabel, rasterize_tree_topology

topo_maps = [rasterize_tree_topology(tree, expand_labels_by=10, bridge_gap_smaller_than=50) for tree in trees_gt]
(art_labels, art_topo), (vei_labels, vei_topo) = topo_maps

m = Mosaic(
    (2, 3),
    cols_titles=["VTree", "Branch labels", "Topology map"],
    rows_titles=["Art.", "Vein"],
    cell_height=400,
    background=fundus.image,
)
draw_tree(trees_gt[0], view=m[0, 0], artery=True, edge_labels=True)
m[0, 0].add_label(fundus_gt.av == 1, colormap="red", opacity=0.2)
m[0, 1].add_image(TopologicalLabel.map_to_rgb(art_labels))
m[0, 2].add_image(np.repeat(art_topo[:, :, None], 3, axis=2))
fundus_gt.draw(view=m[1, 0])
draw_tree(trees_gt[1], view=m[1, 0], artery=False, edge_labels=True)
m[1, 1].add_image(TopologicalLabel.map_to_rgb(vei_labels))
m[1, 2].add_image(np.repeat(vei_topo[:, :, None], 3, axis=2))
m

GridBox(children=(HTML(value='<span/>'), HTML(value='<h3 style="text-align: center;">VTree</h3>'), HTML(value=…

In [6]:
m = Mosaic(
    2,
    cols_titles=["Art.", "Vein"],
    cell_height=400,
    background=fundus.image,
)
m[0].add_image(TopologicalLabel.map_to_rgb(art_labels))
draw_tree(trees[0], view=m[0], artery=True, edge_labels=True)
m[1].add_image(np.repeat(vei_topo[:, :, None], 3, axis=2))
draw_tree(trees[1], view=m[1], artery=False, edge_labels=True)
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">Art.</h3>'), HTML(value='<h3 style="text-align: …

In [7]:
from fundus_vessels_toolkit.segment_to_graph.av_map_fixing import evaluate_branch_direction

np.where(evaluate_branch_direction(trees[1], vei_topo) < 0)[0]

array([60, 84])

In [ ]:
from fundus_vessels_toolkit.segment_to_graph.models.training import deteriorate_segmentation

fundus_gt2 = fundus_gt.update(av=deteriorate_segmentation(fundus_gt))
trees_gt2 = naive_av2tree(fundus_gt2)

m = Mosaic(2, cols_titles=["Predicted", "Ground Truth"], cell_height=400)
fundus_gt.draw(view=m[0])
draw_trees(trees_gt, edge="skeleton", view=m[0])
fundus_gt2.draw(view=m[1])
draw_trees(trees_gt2, view=m[1])
m

Dropped 1 segments from branch 0
Dropped 1 segments from branch 1
Dropped 0 segments from branch 2
Dropped 0 segments from branch 3
Dropped 1 segments from branch 4
Dropped 1 segments from branch 5
Dropped 0 segments from branch 6
Dropped 0 segments from branch 7
Dropped 0 segments from branch 8
Dropped 1 segments from branch 9
Dropped 2 segments from branch 10
Dropped 1 segments from branch 11
Dropped 0 segments from branch 12
Dropped 0 segments from branch 13
Dropped 0 segments from branch 14
Dropped 1 segments from branch 15
Dropped 1 segments from branch 16
Dropped 0 segments from branch 17
Dropped entire branch 18
Dropped 1 segments from branch 18
Dropped 1 segments from branch 19
Dropped 1 segments from branch 20
Dropped 0 segments from branch 21
Dropped 1 segments from branch 24
Dropped 0 segments from branch 25
Dropped 1 segments from branch 26
Dropped 1 segments from branch 27
Dropped 1 segments from branch 28
Dropped 1 segments from branch 29
Dropped 0 segments from branch 30

GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…